# 2021 指数变更与 Stage1 训练域相对收益分布

固定旧域、过渡未知、新训练域和复用评价域；本卡只做 Qlib 价格诊断，不建立交易规则。


In [1]:
import os,json,hashlib,tempfile,subprocess,shlex,tarfile
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
import sys
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import archive_nas_recorder,_settings
card=root/'research/stage1_index_transition_domain_diagnostic_20261023'
plan=json.loads((card/'plan.json').read_text());assert plan['frozen']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/'research/stage1_semiconductor_index_regime_source_20261023/result.json')==plan['sources']['official_index_result_sha256']
assert sha(root/'research/stage1_domain_heads_20261018/summary.json')==plan['sources']['original_Stage1_summary_sha256']
assert sha(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices/qlib.yaml')==plan['sources']['Qlib_price_config_sha256']
assert all(plan['budget'][k]==0 for k in ('new_source_HTTP','model_fits','native_accounts','bootstraps'))
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(root/plan['sources']['Qlib_price_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2018-01-02',end_time='2026-10-09',freq='day'))
assert len(calendar)==2205 and calendar[0]==pd.Timestamp('2018-01-02') and calendar[-1]==pd.Timestamp('2026-10-09')
f=D.features(['SOXL','SOXX'],['Ref($open,-1)/$open-1'],start_time='2018-01-02',end_time='2026-10-08',freq='day')
assert f.shape[1]==1 and f.index.names==['instrument','datetime'],f.index.names
r=f.iloc[:,0].unstack('instrument').rename(columns={'SOXL':'SOXL_open_next_return','SOXX':'SOXX_open_next_return'})
assert list(r.columns)==['SOXL_open_next_return','SOXX_open_next_return'] and r.index.equals(calendar[:-1]) and r.notna().all().all()
r['SOXL_minus_SOXX']=r['SOXL_open_next_return']-r['SOXX_open_next_return']
r['SOXL_minus_3x_SOXX']=r['SOXL_open_next_return']-3*r['SOXX_open_next_return']
regions={'old_training':('2018-01-02','2021-06-11'),'transition_unknown':('2021-06-14','2021-08-27'),'new_training':('2021-08-30','2022-12-30'),'reused_held':('2023-01-03','2026-10-08')}
assert regions=={k:tuple(v.split('..')) for k,v in plan['fixed_domains'].items()}
metrics={}
for name,(start,end) in regions.items():
 x=r.loc[start:end];assert len(x)>0
 metrics[name]={'n':len(x),'first':str(x.index[0].date()),'last':str(x.index[-1].date()),'SOXX_return_mean':float(x['SOXX_open_next_return'].mean())}
 for coordinate in ['SOXL_minus_SOXX','SOXL_minus_3x_SOXX']:
  metrics[name][coordinate+'_mean']=float(x[coordinate].mean())
  metrics[name][coordinate+'_median']=float(x[coordinate].median())
  metrics[name][coordinate+'_p05']=float(x[coordinate].quantile(0.05))
  if coordinate=='SOXL_minus_3x_SOXX':metrics[name][coordinate+'_std']=float(x[coordinate].std())
comparisons={}
for key,left,right in [('new_training_minus_old_training','new_training','old_training'),('reused_held_minus_new_training','reused_held','new_training')]:
 comparisons[key]={metric:metrics[left][metric]-metrics[right][metric] for metric in metrics[left] if metric.endswith(('_mean','_median','_p05'))}
report={'decision':'read_only_2021_index_transition_domain_distribution_attribution_no_strategy_rule','plan_sha256':sha(card/'plan.json'),'Qlib_calendar_days':len(calendar),'mature_price_days':len(r),'domains':metrics,'pairwise':comparisons,'source_event_not_exact_transition_clock':True,'reused_history_not_independent_validation':True,'Stage1_full_account_replayed':False,'new_model_fits':0,'new_native_accounts':0,'objective_achieved':False}
static=pd.concat({'feature':r[['SOXL_open_next_return','SOXX_open_next_return','SOXL_minus_SOXX','SOXL_minus_3x_SOXX']]},axis=1)
static.index.name='datetime';static['instrument']='PAIR';static=static.set_index('instrument',append=True)
ds=DatasetH(handler=DataHandlerLP.from_df(static),segments=regions)
assert sum(len(ds.prepare(name,col_set=['feature'])) for name in regions)==len(r)
tracking=card/'qlib_recorder';assert not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id)
 ds.config(dump_all=True,recursive=True)
 rec.save_objects(artifact_path='inputs',plan=plan,dataset=ds)
 rec.save_objects(artifact_path='analysis',domain_daily=r,report=report)
 rec.log_metrics(mature_price_days=len(r),new_model_fits=0,new_native_accounts=0)
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'result':'result.json','new_model_fits':0,'new_native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['This Qlib price provider is a diagnostic price proxy and does not reconstruct the full original Stage1 09:47 account.','Official SEC notices show an around-date transition, not exact implementation instant.','A domain distribution shift is descriptive and cannot prove causal source of Stage1 loss or a predictive trade rule.','All outcome history is reused; independent forward evidence remains zero.']}
(card/'result.json').write_text(json.dumps(report,indent=2)+'\n')
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json'
binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
e=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='stage1-index-domain-') as t:
 archive=Path(t)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(e['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==e['sha256']
 with tarfile.open(archive,'r:gz') as z:assert any(m.name.endswith('/artifacts/analysis/report') for m in z.getmembers())
summary['nas_evidence']=e;summary['nas_readback']={'verified':True,'sha256':e['sha256']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps({'n':{k:v['n'] for k,v in metrics.items()},'comparisons':comparisons,'recorder_id':rid,'nas_sha256':e['sha256']},indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[18624:MainThread](2026-10-23 07:27:56,353) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[18624:MainThread](2026-10-23 07:27:56,355) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[18624:MainThread](2026-10-23 07:27:56,355) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


[18624:MainThread](2026-10-23 07:27:56,378) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


...

/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[18624:MainThread](2026-10-23 07:27:56,386) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name stage1_index_transition_domain_diagnostic_20261023.


[18624:MainThread](2026-10-23 07:27:56,388) INFO - qlib.workflow - [exp.py:258] - Experiment 895876616621403973 starts running ...


[18624:MainThread](2026-10-23 07:27:56,441) INFO - qlib.workflow - [recorder.py:345] - Recorder f1b34ec1665f4107be8912d0bb6d08c1 starts running under Experiment 895876616621403973 ...


[18624:MainThread](2026-10-23 07:27:57,132) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{
  "n": {
    "old_training": 867,
    "transition_unknown": 54,
    "new_training": 338,
    "reused_held": 945
  },
  "comparisons": {
    "new_training_minus_old_training": {
      "SOXX_return_mean": -0.0018258643103763461,
      "SOXL_minus_SOXX_mean": -0.00338540505617857,
      "SOXL_minus_SOXX_median": -0.0031319260597229004,
      "SOXL_minus_SOXX_p05": -0.019739860296249398,
      "SOXL_minus_3x_SOXX_mean": 0.0002663232880877331,
      "SOXL_minus_3x_SOXX_median": 0.0002447664737701416,
      "SOXL_minus_3x_SOXX_p05": 0.0010656386613845815
    },
    "reused_held_minus_new_training": {
      "SOXX_return_mean": 0.002465082099661231,
      "SOXL_minus_SOXX_mean": 0.004546476760879159,
      "SOXL_minus_SOXX_median": 0.002927243709564209,
      "SOXL_minus_SOXX_p05": 0.011382395029067988,
      "SOXL_minus_3x_SOXX_mean": -0.00038368730747606605,
      "SOXL_minus_3x_SOXX_median": -0.000342637300491333,
      "SOXL_minus_3x_SOXX_p05": 0.002479639649391175
    }
  },
  "recorder

# 2021 指数变更与 Stage1 训练域的相对收益分布

## 中文结论

官方原件显示 SOXX 与 SOXL 在 2021 年夏季先后更换底层指数，但未给出足以硬切某个交易秒点的证据。本卡事先把 2018-01-02 至 2021-06-11 定为旧训练域，2021-06-14 至 08-27 保留为过渡未知，2021-08-30 至 2022-12-30 定为共同新指数训练域，2023-01-03 至 2026-10-08 是复用评价域。用原生 Qlib 日历、价格、DatasetH、Recorder 读取 2,204 个成熟日线开盘到次日开盘收益，四域分别有 867、54、338、945 日；不拟合模型、不跑账户。

| 域 | SOXX 日均收益 | SOXL−SOXX 日均收益 | SOXL−3×SOXX 日均残差 |
|---|---:|---:|---:|
| 旧训练域 | +0.1331% | +0.2184% | −0.0477% |
| 过渡未知 | +0.1573% | +0.2631% | −0.0516% |
| 新训练域 | −0.0495% | −0.1201% | −0.0211% |
| 复用评价域 | +0.1970% | +0.3346% | −0.0595% |

新训练域相对旧训练域，SOXL−SOXX 日均差下降 **0.3385 个百分点**；同期 2×SOXX 日均收益下降 **0.3652 个百分点**，而 SOXL−3×SOXX 残差改善 **0.0266 个百分点**。新训练域到复用评价域，相对差回升 **0.4546 个百分点**，对应 2×SOXX 回升 **0.4930 个百分点**，残差反而下降 **0.0384 个百分点**。这些是恒等式 `SOXL−SOXX = 2×SOXX + (SOXL−3×SOXX)` 的同口径描述。2022 熊市与随后市场方向差异足以解释均值变化的主要部分；当前数据**不支持把指数更换直接认定为 Stage1 失败根因**。

这只是日线价格代理的域诊断，并非完整 Stage1 09:47 账户、风险状态或费用收益；四段也不是独立验证样本。指数变更仍可能影响成分、跟踪误差尾部或模型输入，但本结果没有给出可提前观察的收益信号，更不能按事后分段切换策略。原目标的完整 Stage1 0bp 净值尚未超过 SOXX，**不推广**。下一项机制应提供区别于 SOXX 本身日收益、在 09:47 前可获得且有历史时点证据的新信息，并先检验其全区间增量，再进入完整账户。

Qlib Recorder `f1b34ec1665f4107be8912d0bb6d08c1` 已完成读回；原生 DatasetH 2,204 行与报告一致。原始 Qlib Recorder 已归档 Yanxuan NAS 并按哈希冷读，独立复核结果见 `independent_review.json`。
